# ProvideQ Web RAG Evaluation Dashboard

This notebook visualizes evaluation results for the Web RAG evidence retrieval module.

It expects that the evaluation pipeline has already produced a comparison folder containing:

- `comparison_aggregate_wide.csv`
- `comparison_aggregate_long.csv`
- `comparison_aggregate_deltas.csv`
- `comparison_question_wide.csv`

The notebook does not rerun retrieval or metric computation. It only loads saved reports and visualizes them.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

# If you start Jupyter from the project root, this is correct.
# If you start it from notebooks/, the code will automatically use the parent folder.
current_dir = Path.cwd()

if (current_dir / "outputs").exists():
    PROJECT_ROOT = current_dir
elif current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

COMPARISON_DIR = PROJECT_ROOT / "outputs" / "evaluation" / "comparison_provideq20"

AGGREGATE_WIDE_PATH = COMPARISON_DIR / "comparison_aggregate_wide.csv"
AGGREGATE_LONG_PATH = COMPARISON_DIR / "comparison_aggregate_long.csv"
DELTAS_PATH = COMPARISON_DIR / "comparison_aggregate_deltas.csv"
QUESTION_WIDE_PATH = COMPARISON_DIR / "comparison_question_wide.csv"

print("Project root:", PROJECT_ROOT)
print("Comparison directory:", COMPARISON_DIR)

Project root: c:\Users\taymm\Desktop\ProvideQ\provideq-web-rag
Comparison directory: c:\Users\taymm\Desktop\ProvideQ\provideq-web-rag\outputs\evaluation\comparison_provideq20


## 1. Load comparison reports

In [ ]:
def load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    if path.stat().st_size == 0:
        print(f"Warning: {path.name} is empty; returning an empty DataFrame.")
        return pd.DataFrame()

    return pd.read_csv(path)


aggregate_wide = load_csv(AGGREGATE_WIDE_PATH)
aggregate_long = load_csv(AGGREGATE_LONG_PATH)
deltas = load_csv(DELTAS_PATH)
question_wide = load_csv(QUESTION_WIDE_PATH)

print("aggregate_wide:", aggregate_wide.shape)
print("aggregate_long:", aggregate_long.shape)
print("deltas:", deltas.shape)
print("question_wide:", question_wide.shape)

EmptyDataError: No columns to parse from file

## 2. Quick overview tables

In [ ]:
aggregate_wide

In [3]:
deltas.sort_values(["metric_name", "k", "run_label"])

NameError: name 'deltas' is not defined

## 3. Metric curves over k

These plots show how each metric changes as more retrieved snippets are allowed into the evidence pack.

In [ ]:
METRICS = [
    "ROUGE1_Nugget@k",
    "ROUGEL_Nugget@k",
    "ROUGE_Nugget@k",
    "BM25_Nugget@k",
    "SemanticNuggetMatch@k",
    "SemanticAnswerMatch@k",
]

aggregate_long["k"] = aggregate_long["k"].astype(int)
aggregate_long["metric_value"] = pd.to_numeric(aggregate_long["metric_value"], errors="coerce")


def plot_metric_curve(metric_name: str) -> None:
    data = aggregate_long[aggregate_long["metric_name"] == metric_name].copy()
    data = data.dropna(subset=["metric_value"])

    if data.empty:
        print(f"No data for metric: {metric_name}")
        return

    plt.figure(figsize=(8, 5))

    for run_label, group in data.groupby("run_label"):
        group = group.sort_values("k")
        plt.plot(group["k"], group["metric_value"], marker="o", label=run_label)

    plt.title(metric_name)
    plt.xlabel("k")
    plt.ylabel("score")
    plt.xticks(sorted(data["k"].unique()))
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()


for metric in METRICS:
    plot_metric_curve(metric)

## 4. Delta plots against the baseline

Positive values mean the tested run improved over the baseline. Negative values mean it performed worse.

In [ ]:
deltas["k"] = deltas["k"].astype(int)
deltas["delta"] = pd.to_numeric(deltas["delta"], errors="coerce")


def plot_metric_delta(metric_name: str) -> None:
    data = deltas[deltas["metric_name"] == metric_name].copy()
    data = data.dropna(subset=["delta"])

    if data.empty:
        print(f"No delta data for metric: {metric_name}")
        return

    plt.figure(figsize=(8, 5))

    for run_label, group in data.groupby("run_label"):
        group = group.sort_values("k")
        plt.plot(group["k"], group["delta"], marker="o", label=run_label)

    plt.axhline(0, linewidth=1)
    plt.title(f"Delta vs baseline: {metric_name}")
    plt.xlabel("k")
    plt.ylabel("delta")
    plt.xticks(sorted(data["k"].unique()))
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()


for metric in METRICS:
    plot_metric_delta(metric)

## 5. Best and worst aggregate changes

This table shows where the new method improved most and where it became worse.

In [ ]:
deltas_sorted = deltas.sort_values("delta", ascending=False)
deltas_sorted[[
    "baseline_label",
    "run_label",
    "k",
    "metric_name",
    "baseline_value",
    "run_value",
    "delta",
    "relative_change_percent",
]]

## 6. Per-question comparison

This section compares two runs question by question for a selected metric and k value. It helps identify which benchmark questions improved and which got worse.

In [ ]:
# Change these values depending on what you want to inspect.
BASELINE_LABEL = "lexical"
IMPROVED_LABEL = "medcpt_hybrid"
SELECTED_K = 5
SELECTED_METRIC = "SemanticNuggetMatch@k"

question_wide["k"] = question_wide["k"].astype(int)

baseline_questions = question_wide[
    (question_wide["run_label"] == BASELINE_LABEL)
    & (question_wide["k"] == SELECTED_K)
].copy()

improved_questions = question_wide[
    (question_wide["run_label"] == IMPROVED_LABEL)
    & (question_wide["k"] == SELECTED_K)
].copy()

baseline_questions[SELECTED_METRIC] = pd.to_numeric(
    baseline_questions[SELECTED_METRIC], errors="coerce"
)
improved_questions[SELECTED_METRIC] = pd.to_numeric(
    improved_questions[SELECTED_METRIC], errors="coerce"
)

question_comparison = baseline_questions[["question_id", "question", SELECTED_METRIC]].merge(
    improved_questions[["question_id", SELECTED_METRIC]],
    on="question_id",
    suffixes=("_baseline", "_improved"),
)

question_comparison["delta"] = (
    question_comparison[f"{SELECTED_METRIC}_improved"]
    - question_comparison[f"{SELECTED_METRIC}_baseline"]
)

question_comparison = question_comparison.sort_values("delta", ascending=False)
question_comparison

## 7. Top improvements and regressions

In [ ]:
top_n = 10

top_improvements = question_comparison.head(top_n)
top_regressions = question_comparison.tail(top_n).sort_values("delta")

print("Top improvements")
display(top_improvements)

print("Top regressions")
display(top_regressions)

In [ ]:
def plot_question_deltas(data: pd.DataFrame, title: str) -> None:
    if data.empty:
        print("No data to plot.")
        return

    plot_data = data.copy().sort_values("delta")

    plt.figure(figsize=(10, 6))
    plt.barh(plot_data["question_id"], plot_data["delta"])
    plt.axvline(0, linewidth=1)
    plt.title(title)
    plt.xlabel("delta")
    plt.ylabel("question_id")
    plt.grid(True, axis="x", alpha=0.3)
    plt.show()


plot_question_deltas(
    top_improvements,
    f"Top improvements: {SELECTED_METRIC} @ k={SELECTED_K}",
)

plot_question_deltas(
    top_regressions,
    f"Top regressions: {SELECTED_METRIC} @ k={SELECTED_K}",
)

## 8. Thesis interpretation checklist

Use the tables and plots above to answer these questions:

1. Does the improved ranking method increase nugget coverage compared to the baseline?
2. Does the improvement appear at early ranks such as k=1 or k=3, or only when k is larger?
3. Do lexical metrics and semantic metrics agree?
4. Which questions improved most?
5. Which questions became worse, and why?
6. Are regressions caused by retrieval failure, ranking failure, missing abstracts, or weak gold nuggets?

For the thesis, the most useful analysis is not only the average score. The per-question regressions are important because they show where the Web RAG pipeline still fails.